In [1]:
learner_profiles_bronze_df = spark.table(
    "demo.bronze.learner_profiles"
)

learner_profiles_bronze_df.show(truncate=False)
learner_profiles_bronze_df.printSchema()

+--------+-------------------+-------------------+--------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|user_id |profile_updated_at |ingestion_time     |source_system       |raw_payload                                                                                                                                                                                                              |
+--------+-------------------+-------------------+--------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|user_001|2026-07-01 08:00:00|2026-07-01 08:05:00|application_database|{"declared_background_level":"Beginner","learning_goal":"Im

In [2]:
from pyspark.sql.types import StructType, StructField, StringType

learner_profile_payload_schema = StructType([
    StructField("declared_background_level", StringType(), True),
    StructField("learning_goal", StringType(), True),
    StructField("main_domain", StringType(), True),
    StructField("preferred_language", StringType(), True),
    StructField("registration_date", StringType(), True),
])

In [3]:
from pyspark.sql.functions import col, from_json

learner_profiles_parsed_df = (
    learner_profiles_bronze_df
    .withColumn(
        "payload",
        from_json(
            col("raw_payload"),
            learner_profile_payload_schema
        )
    )
)

learner_profiles_parsed_df.select(
    "user_id",
    "profile_updated_at",
    "payload"
).show(truncate=False)

learner_profiles_parsed_df.printSchema()

+--------+-------------------+----------------------------------------------------------------------------------------------------------------+
|user_id |profile_updated_at |payload                                                                                                         |
+--------+-------------------+----------------------------------------------------------------------------------------------------------------+
|user_001|2026-07-01 08:00:00|{Beginner, Improve understanding of operating systems, Computer Science, English, 2026-07-01}                   |
|user_001|2026-07-23 18:10:00|{Intermediate, Strengthen virtual memory and operating systems knowledge, Computer Science, English, 2026-07-01}|
|user_002|2026-07-03 09:30:00|{Intermediate, Improve programming fundamentals and recursion, Computer Science, English, 2026-07-03}           |
|user_003|2026-07-05 11:00:00|{Intermediate, Learn technical concepts through visual explanations, Computer Science, English, 2026-07-05

In [4]:
from pyspark.sql.functions import (
    col,
    to_date,
    trim,
    initcap,
    row_number,
    when
)
from pyspark.sql.window import Window

profile_window = (
    Window
    .partitionBy("user_id")
    .orderBy(col("profile_updated_at").desc())
)

learner_profiles_silver_df = (
    learner_profiles_parsed_df
    .select(
        col("user_id"),
        to_date(col("payload.registration_date")).alias("registration_date"),
        trim(col("payload.preferred_language")).alias("preferred_language"),
        initcap(
            trim(col("payload.declared_background_level"))
        ).alias("background_level"),
        trim(col("payload.learning_goal")).alias("learning_goal"),
        trim(col("payload.main_domain")).alias("main_domain"),
        col("profile_updated_at"),
        col("ingestion_time")
    )
    .withColumn(
        "profile_rank",
        row_number().over(profile_window)
    )
    .withColumn(
        "is_current",
        when(col("profile_rank") == 1, True).otherwise(False)
    )
    .drop("profile_rank")
)

learner_profiles_silver_df.show(truncate=False)
learner_profiles_silver_df.printSchema()

+--------+-----------------+------------------+----------------+---------------------------------------------------------+----------------+-------------------+-------------------+----------+
|user_id |registration_date|preferred_language|background_level|learning_goal                                            |main_domain     |profile_updated_at |ingestion_time     |is_current|
+--------+-----------------+------------------+----------------+---------------------------------------------------------+----------------+-------------------+-------------------+----------+
|user_001|2026-07-01       |English           |Intermediate    |Strengthen virtual memory and operating systems knowledge|Computer Science|2026-07-23 18:10:00|2026-07-23 18:15:00|true      |
|user_001|2026-07-01       |English           |Beginner        |Improve understanding of operating systems               |Computer Science|2026-07-01 08:00:00|2026-07-01 08:05:00|false     |
|user_002|2026-07-03       |English          

In [5]:
spark.table(
    "demo.silver.learner_profiles"
).count()

0

In [6]:
learner_profiles_silver_df.writeTo(
    "demo.silver.learner_profiles"
).append()

In [7]:
spark.sql("""
SELECT COUNT(*) AS row_count
FROM demo.silver.learner_profiles
""").show()

+---------+
|row_count|
+---------+
|        4|
+---------+



In [8]:
spark.sql("""
SELECT *
FROM demo.silver.learner_profiles
ORDER BY user_id, profile_updated_at
""").show(truncate=False)

+--------+-----------------+------------------+----------------+---------------------------------------------------------+----------------+-------------------+-------------------+----------+
|user_id |registration_date|preferred_language|background_level|learning_goal                                            |main_domain     |profile_updated_at |ingestion_time     |is_current|
+--------+-----------------+------------------+----------------+---------------------------------------------------------+----------------+-------------------+-------------------+----------+
|user_001|2026-07-01       |English           |Beginner        |Improve understanding of operating systems               |Computer Science|2026-07-01 08:00:00|2026-07-01 08:05:00|false     |
|user_001|2026-07-01       |English           |Intermediate    |Strengthen virtual memory and operating systems knowledge|Computer Science|2026-07-23 18:10:00|2026-07-23 18:15:00|true      |
|user_002|2026-07-03       |English          

In [9]:
question_bank_bronze_df = spark.table(
    "demo.bronze.question_bank"
)

question_bank_bronze_df.show(truncate=False)
question_bank_bronze_df.printSchema()

+------------+----------------+---------------------------+-------------------+-------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|question_id |question_version|source_system              |created_at         |ingestion_time     |raw_payload                                             

In [10]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    BooleanType
)

question_payload_schema = StructType([
    StructField("content_hash", StringType(), True),
    StructField("correct_option_letter", StringType(), True),
    StructField("created_by", StringType(), True),
    StructField("difficulty_level", IntegerType(), True),
    StructField("domain", StringType(), True),
    StructField("generation_model", StringType(), True),
    StructField("is_active", BooleanType(), True),
    StructField("option_a_text", StringType(), True),
    StructField("option_b_text", StringType(), True),
    StructField("option_c_text", StringType(), True),
    StructField("option_d_text", StringType(), True),
    StructField("question_text", StringType(), True),
    StructField("question_type", StringType(), True),
    StructField("subtopic", StringType(), True),
    StructField("topic", StringType(), True),
    StructField("validation_status", StringType(), True),
])

In [11]:
from pyspark.sql.functions import col, from_json

question_bank_parsed_df = (
    question_bank_bronze_df
    .withColumn(
        "payload",
        from_json(
            col("raw_payload"),
            question_payload_schema
        )
    )
)

question_bank_parsed_df.select(
    "question_id",
    "question_version",
    "created_at",
    "payload"
).show(truncate=False)

question_bank_parsed_df.printSchema()

+------------+----------------+-------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|question_id |question_version|created_at         |payload                                                                                                                                                                                                                                                                                                                                                                                                                    

In [12]:
from pyspark.sql.functions import col, trim, upper, lower

question_bank_silver_df = (
    question_bank_parsed_df
    .select(
        col("question_id"),
        col("question_version"),
        trim(col("payload.question_text")).alias("question_text"),
        lower(trim(col("payload.question_type"))).alias("question_type"),
        trim(col("payload.domain")).alias("domain"),
        trim(col("payload.topic")).alias("topic"),
        trim(col("payload.subtopic")).alias("subtopic"),
        col("payload.difficulty_level").alias("difficulty_level"),
        trim(col("payload.option_a_text")).alias("option_a_text"),
        trim(col("payload.option_b_text")).alias("option_b_text"),
        trim(col("payload.option_c_text")).alias("option_c_text"),
        trim(col("payload.option_d_text")).alias("option_d_text"),
        upper(
            trim(col("payload.correct_option_letter"))
        ).alias("correct_option_letter"),
        trim(col("payload.created_by")).alias("created_by"),
        trim(col("payload.generation_model")).alias("generation_model"),
        col("created_at"),
        col("payload.is_active").alias("is_active"),
        lower(
            trim(col("payload.validation_status"))
        ).alias("validation_status"),
        trim(col("payload.content_hash")).alias("content_hash")
    )
)

question_bank_silver_df.show(truncate=False)
question_bank_silver_df.printSchema()

+------------+----------------+-------------------------------------------------------------------+---------------+----------------+-----------------+---------------------+----------------+-----------------------------------------------------------+-----------------------------------------------------------------------------------+------------------------------------------------------------+-------------------------------------------------------------------+---------------------+----------+-------------------------------+-------------------+---------+-----------------+----------------------------------------------------------------+
|question_id |question_version|question_text                                                      |question_type  |domain          |topic            |subtopic             |difficulty_level|option_a_text                                              |option_b_text                                                                      |option_c_text             

In [13]:
spark.table(
    "demo.silver.question_bank"
).count()

0

In [14]:
question_bank_silver_df.writeTo(
    "demo.silver.question_bank"
).append()

In [15]:
spark.sql("""
SELECT COUNT(*) AS row_count
FROM demo.silver.question_bank
""").show()

+---------+
|row_count|
+---------+
|        5|
+---------+



In [16]:
spark.sql("""
SELECT
    question_id,
    question_version,
    question_text,
    difficulty_level,
    correct_option_letter,
    validation_status,
    is_active
FROM demo.silver.question_bank
ORDER BY question_id, question_version
""").show(truncate=False)

+------------+----------------+-------------------------------------------------------------------+----------------+---------------------+-----------------+---------+
|question_id |question_version|question_text                                                      |difficulty_level|correct_option_letter|validation_status|is_active|
+------------+----------------+-------------------------------------------------------------------+----------------+---------------------+-----------------+---------+
|question_001|1               |What causes a page fault?                                          |2               |B                    |approved         |true     |
|question_002|1               |What is the main purpose of virtual memory?                        |2               |B                    |approved         |true     |
|question_003|1               |What is the purpose of a base case in recursion?                   |2               |A                    |approved         |true     

In [17]:
spark.table(
    "demo.silver.question_bank"
).count()

5

In [18]:
reference_materials_bronze_df = spark.table(
    "demo.bronze.reference_materials"
)

reference_materials_bronze_df.show(truncate=False)
reference_materials_bronze_df.printSchema()

+-------------+--------------------+----------------------+-----------------------------------+-------------------------------+-------------------+-------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|reference_id |batch_id            |source_type           |source_name                        |file_name                      |import_time        |ingestion_time     |raw_payload                                                                                                                                                                                  

In [19]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType
)

reference_material_payload_schema = StructType([
    StructField("author_or_owner", StringType(), True),
    StructField("content_text", StringType(), True),
    StructField("domain", StringType(), True),
    StructField("page_number", IntegerType(), True),
    StructField("reliability_level", StringType(), True),
    StructField("section_name", StringType(), True),
    StructField("subtopic", StringType(), True),
    StructField("title", StringType(), True),
    StructField("topic", StringType(), True),
])

In [20]:
from pyspark.sql.functions import col, from_json

reference_materials_parsed_df = (
    reference_materials_bronze_df
    .withColumn(
        "payload",
        from_json(
            col("raw_payload"),
            reference_material_payload_schema
        )
    )
)

reference_materials_parsed_df.select(
    "reference_id",
    "file_name",
    "import_time",
    "payload"
).show(truncate=False)

reference_materials_parsed_df.printSchema()

+-------------+-------------------------------+-------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|reference_id |file_name                      |import_time        |payload                                                                                                                                                                                                                                                                                                                                                                         |
+-------------+-------------------------------+-------------------+-----------------------------------------------------------

In [21]:
from pyspark.sql.functions import (
    col,
    trim,
    lower,
    lit,
    sha2,
    concat_ws
)

reference_materials_silver_df = (
    reference_materials_parsed_df
    .select(
        col("reference_id"),
        col("batch_id"),
        trim(col("source_type")).alias("source_type"),
        trim(col("source_name")).alias("source_name"),
        trim(col("file_name")).alias("file_name"),
        col("import_time"),
        col("ingestion_time"),
        trim(col("payload.domain")).alias("domain"),
        trim(col("payload.title")).alias("title"),
        trim(col("payload.topic")).alias("topic"),
        trim(col("payload.content_text")).alias("content_text"),
        lower(
            trim(col("payload.reliability_level"))
        ).alias("reliability_level"),
        trim(col("payload.author_or_owner")).alias("author_or_owner")
    )
    .withColumn(
        "content_hash",
        sha2(
            concat_ws(
                "||",
                col("title"),
                col("content_text"),
                col("author_or_owner")
            ),
            256
        )
    )
    .withColumn(
        "is_active",
        lit(True)
    )
)

reference_materials_silver_df.show(truncate=False)
reference_materials_silver_df.printSchema()

+-------------+--------------------+----------------------+-----------------------------------+-------------------------------+-------------------+-------------------+----------------+---------------------------------+-----------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------+------------------------------+----------------------------------------------------------------+---------+
|reference_id |batch_id            |source_type           |source_name                        |file_name                      |import_time        |ingestion_time     |domain          |title                            |topic            |content_text                                                                                                                                                    

In [22]:
spark.table(
    "demo.silver.reference_materials"
).count()

0

In [23]:
reference_materials_silver_df.writeTo(
    "demo.silver.reference_materials"
).append()

In [24]:
spark.sql("""
SELECT COUNT(*) AS row_count
FROM demo.silver.reference_materials
""").show()

+---------+
|row_count|
+---------+
|        6|
+---------+



In [25]:
spark.sql("""
SELECT
    reference_id,
    source_type,
    title,
    topic,
    reliability_level,
    content_hash,
    is_active
FROM demo.silver.reference_materials
ORDER BY reference_id
""").show(truncate=False)

+-------------+----------------------+---------------------------------+-----------------+-----------------+----------------------------------------------------------------+---------+
|reference_id |source_type           |title                            |topic            |reliability_level|content_hash                                                    |is_active|
+-------------+----------------------+---------------------------------+-----------------+-----------------+----------------------------------------------------------------+---------+
|reference_001|lecture_slides        |Page Fault Definition            |Operating Systems|official         |70e24d35d9aa107109f437640d3fa5bbaca7d7678a30692eb7741aee100e5d3c|true     |
|reference_002|lecture_slides        |Page Fault Handling Steps        |Operating Systems|official         |b36f7e6a6fedf7f949be880459b98bde8f7d978d827d44dd87079b357c25a45b|true     |
|reference_003|official_documentation|Purpose of Virtual Memory        |Operatin

In [26]:
learning_events_bronze_df = spark.table(
    "demo.bronze.learning_events"
)

learning_events_bronze_df.show(truncate=False)
learning_events_bronze_df.printSchema()

+--------+--------+-----------+-----------------------+-------------------+-------------------+-------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|event_id|user_id |session_id |event_type             |event_time         |ingestion_time     |source_system|raw_p

In [27]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    BooleanType,
    ArrayType
)

answer_schema = StructType([
    StructField("attempt_duration_seconds", IntegerType(), True),
    StructField("hints_used", IntegerType(), True),
    StructField("question_id", StringType(), True),
    StructField("question_version", IntegerType(), True),
    StructField("selected_option_letter", StringType(), True),
])

learning_event_payload_schema = StructType([
    StructField("answers", ArrayType(answer_schema), True),
    StructField("assistant_answer", StringType(), True),
    StructField("completion_status", StringType(), True),
    StructField("conversation_id", StringType(), True),
    StructField("conversation_summary", StringType(), True),
    StructField("detected_concepts", ArrayType(StringType()), True),
    StructField("difficulty_indicators", ArrayType(StringType()), True),
    StructField("important_points", ArrayType(StringType()), True),
    StructField("learner_intent", StringType(), True),
    StructField("possible_confusion", BooleanType(), True),
    StructField("practice_id", StringType(), True),
    StructField("processing_model", StringType(), True),
    StructField("started_at", StringType(), True),
    StructField("submitted_at", StringType(), True),
    StructField("topic_id", StringType(), True),
    StructField("user_prompt", StringType(), True),
])

In [28]:
from pyspark.sql.functions import (
    col,
    from_json,
    to_date,
    hour,
    lower,
    trim,
    when,
    lit
)

learning_events_silver_df = (
    learning_events_bronze_df
    .withColumn(
        "parsed_payload",
        from_json(
            col("raw_payload"),
            learning_event_payload_schema
        )
    )
    .select(
        col("event_id"),
        col("user_id"),
        col("session_id"),
        lower(trim(col("event_type"))).alias("event_type"),
        col("event_time"),
        col("ingestion_time"),
        lower(trim(col("source_system"))).alias("source_system"),
        to_date(col("event_time")).alias("event_date"),
        hour(col("event_time")).alias("event_hour"),
        col("parsed_payload")
    )
    .withColumn(
        "payload_valid",
        col("parsed_payload").isNotNull()
    )
    .withColumn(
        "processing_status",
        when(col("payload_valid"), lit("processed"))
        .otherwise(lit("failed"))
    )
    .drop("parsed_payload")
)

learning_events_silver_df.show(truncate=False)
learning_events_silver_df.printSchema()

+--------+--------+-----------+-----------------------+-------------------+-------------------+-------------+----------+----------+-------------+-----------------+
|event_id|user_id |session_id |event_type             |event_time         |ingestion_time     |source_system|event_date|event_hour|payload_valid|processing_status|
+--------+--------+-----------+-----------------------+-------------------+-------------------+-------------+----------+----------+-------------+-----------------+
|evt_0001|user_001|session_001|ai_learning_interaction|2026-07-20 09:00:05|2026-07-20 09:00:08|chat         |2026-07-20|9         |true         |processed        |
|evt_0002|user_001|session_001|practice_submitted     |2026-07-20 09:12:00|2026-07-20 09:12:03|practice_app |2026-07-20|9         |true         |processed        |
|evt_0003|user_002|session_002|ai_learning_interaction|2026-07-21 11:00:04|2026-07-21 11:00:07|chat         |2026-07-21|11        |true         |processed        |
|evt_0004|user_0

In [29]:
spark.table(
    "demo.silver.learning_events"
).count()

0

In [30]:
learning_events_silver_df.writeTo(
    "demo.silver.learning_events"
).append()

In [31]:
spark.sql("""
SELECT
    event_id,
    event_type,
    event_time,
    source_system,
    event_date,
    event_hour,
    payload_valid,
    processing_status
FROM demo.silver.learning_events
ORDER BY event_time
""").show(truncate=False)

+--------+-----------------------+-------------------+-------------+----------+----------+-------------+-----------------+
|event_id|event_type             |event_time         |source_system|event_date|event_hour|payload_valid|processing_status|
+--------+-----------------------+-------------------+-------------+----------+----------+-------------+-----------------+
|evt_0001|ai_learning_interaction|2026-07-20 09:00:05|chat         |2026-07-20|9         |true         |processed        |
|evt_0002|practice_submitted     |2026-07-20 09:12:00|practice_app |2026-07-20|9         |true         |processed        |
|evt_0003|ai_learning_interaction|2026-07-21 11:00:04|chat         |2026-07-21|11        |true         |processed        |
|evt_0004|practice_submitted     |2026-07-21 11:11:00|practice_app |2026-07-21|11        |true         |processed        |
|evt_0005|ai_learning_interaction|2026-07-22 14:00:05|chat         |2026-07-22|14        |true         |processed        |
|evt_0006|practi

In [32]:
from pyspark.sql.functions import (
    col,
    explode,
    concat_ws,
    sha2,
    lit,
    row_number,
    when,
    to_timestamp
)
from pyspark.sql.window import Window

practice_events_df = (
    learning_events_bronze_df
    .filter(col("event_type") == "practice_submitted")
    .withColumn(
        "parsed_payload",
        from_json(
            col("raw_payload"),
            learning_event_payload_schema
        )
    )
    .withColumn(
        "answer",
        explode(col("parsed_payload.answers"))
    )
)

In [33]:
attempt_window = (
    Window
    .partitionBy(
        "user_id",
        col("answer.question_id")
    )
    .orderBy(
        col("event_time"),
        col("event_id")
    )
)

practice_attempts_silver_df = (
    practice_events_df
    .join(
        spark.table("demo.silver.question_bank").select(
            "question_id",
            "question_version",
            "correct_option_letter"
        ),
        (
            col("answer.question_id") == col("question_id")
        )
        & (
            col("answer.question_version") == col("question_version")
        ),
        "left"
    )
    .withColumn(
        "attempt_number",
        row_number().over(attempt_window)
    )
    .withColumn(
        "is_correct",
        col("answer.selected_option_letter")
        == col("correct_option_letter")
    )
    .withColumn(
        "score",
        when(col("is_correct"), lit(1.0))
        .otherwise(lit(0.0))
        .cast("float")
    )
    .withColumn(
        "attempt_id",
        sha2(
            concat_ws(
                "||",
                col("event_id"),
                col("answer.question_id"),
                col("attempt_number").cast("string")
            ),
            256
        )
    )
    .select(
        col("attempt_id"),
        col("event_id"),
        col("user_id"),
        col("session_id"),
        col("parsed_payload.practice_id").alias("practice_id"),
        col("answer.question_id").alias("question_id"),
        col("answer.question_version").alias("question_version"),
        col("event_time").alias("attempt_time"),
        col("answer.selected_option_letter")
            .alias("selected_option_letter"),
        col("is_correct"),
        col("score"),
        col("answer.hints_used").alias("hints_used"),
        col("answer.attempt_duration_seconds")
            .alias("attempt_duration_seconds"),
        col("attempt_number")
    )
)

practice_attempts_silver_df.show(truncate=False)
practice_attempts_silver_df.printSchema()

+----------------------------------------------------------------+--------+--------+-----------+------------+------------+----------------+-------------------+----------------------+----------+-----+----------+------------------------+--------------+
|attempt_id                                                      |event_id|user_id |session_id |practice_id |question_id |question_version|attempt_time       |selected_option_letter|is_correct|score|hints_used|attempt_duration_seconds|attempt_number|
+----------------------------------------------------------------+--------+--------+-----------+------------+------------+----------------+-------------------+----------------------+----------+-----+----------+------------------------+--------------+
|fb827af1d27682d1cc5d74c5cd4ea504e5514853143ad2d50d9a6406ad89c514|evt_0002|user_001|session_001|practice_001|question_001|1               |2026-07-20 09:12:00|C                     |false     |0.0  |1         |80                      |1           

In [34]:
spark.table(
    "demo.silver.practice_attempts"
).count()

0

In [35]:
practice_attempts_silver_df.writeTo(
    "demo.silver.practice_attempts"
).append()

In [36]:
spark.sql("""
SELECT COUNT(*) AS row_count
FROM demo.silver.practice_attempts
""").show()

+---------+
|row_count|
+---------+
|        5|
+---------+



In [37]:
spark.sql("""
SELECT
    attempt_id,
    event_id,
    user_id,
    practice_id,
    question_id,
    selected_option_letter,
    is_correct,
    score,
    hints_used,
    attempt_duration_seconds,
    attempt_number
FROM demo.silver.practice_attempts
ORDER BY attempt_time, question_id
""").show(truncate=False)

+----------------------------------------------------------------+--------+--------+------------+------------+----------------------+----------+-----+----------+------------------------+--------------+
|attempt_id                                                      |event_id|user_id |practice_id |question_id |selected_option_letter|is_correct|score|hints_used|attempt_duration_seconds|attempt_number|
+----------------------------------------------------------------+--------+--------+------------+------------+----------------------+----------+-----+----------+------------------------+--------------+
|fb827af1d27682d1cc5d74c5cd4ea504e5514853143ad2d50d9a6406ad89c514|evt_0002|user_001|practice_001|question_001|C                     |false     |0.0  |1         |80                      |1             |
|d76fe2bf4801f31255fe5f66181612e6d4d8fa3c5c7ffe736c2ca01bb77c485c|evt_0002|user_001|practice_001|question_002|B                     |true      |1.0  |0         |65                      |1     

In [38]:
learning_feedback_bronze_df = spark.table(
    "demo.bronze.learning_feedback"
)

learning_feedback_bronze_df.show(truncate=False)
learning_feedback_bronze_df.printSchema()

+------------+--------+-----------+------------+----------------+-------------------+-------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|feedback_id |user_id |session_id |practice_id |feedback_stage  |feedback_time      |ingestion_time     |raw_payload                                                                                                                                                                                                                                                                                                                                                                      

In [39]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    BooleanType,
    ArrayType
)

topic_feedback_schema = StructType([
    StructField("confidence_score", IntegerType(), True),
    StructField("perceived_understanding_score", IntegerType(), True),
    StructField("still_confused", BooleanType(), True),
    StructField("topic_id", StringType(), True),
])

learning_feedback_payload_schema = StructType([
    StructField("confidence_after_score", IntegerType(), True),
    StructField("confidence_before_score", IntegerType(), True),
    StructField("expected_difficulty_score", IntegerType(), True),
    StructField("free_text", StringType(), True),
    StructField("free_text_after", StringType(), True),
    StructField("free_text_before", StringType(), True),
    StructField("overall_confidence_score", IntegerType(), True),
    StructField("overall_motivation_score", IntegerType(), True),
    StructField("overall_stress_score", IntegerType(), True),
    StructField("perceived_difficulty_score", IntegerType(), True),
    StructField("perceived_understanding_after_score", IntegerType(), True),
    StructField("perceived_understanding_before_score", IntegerType(), True),
    StructField("still_confused", BooleanType(), True),
    StructField(
        "topics_feedback",
        ArrayType(topic_feedback_schema),
        True
    ),
])

In [40]:
from pyspark.sql.functions import col, from_json

learning_feedback_parsed_df = (
    learning_feedback_bronze_df
    .withColumn(
        "payload",
        from_json(
            col("raw_payload"),
            learning_feedback_payload_schema
        )
    )
)

learning_feedback_parsed_df.select(
    "feedback_id",
    "feedback_stage",
    "practice_id",
    "feedback_time",
    "payload"
).show(truncate=False)

learning_feedback_parsed_df.printSchema()

+------------+----------------+------------+-------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|feedback_id |feedback_stage  |practice_id |feedback_time      |payload                                                                                                                                                                                                              |
+------------+----------------+------------+-------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|feedback_001|before_practice |practice_001|2026-07-20 09:04:00|{NULL, 4, 8, NULL, NULL, I am still confused about page faults and expect the practice to be diffic

In [41]:
from pyspark.sql.functions import (
    col,
    trim,
    round as spark_round,
    unix_timestamp
)

pre_practice_feedback_silver_df = (
    learning_feedback_parsed_df
    .filter(col("feedback_stage") == "before_practice")
    .select(
        col("feedback_id"),
        col("user_id"),
        col("session_id"),
        col("practice_id"),
        col("feedback_time"),
        col("ingestion_time"),

        spark_round(
            (
                unix_timestamp(col("ingestion_time"))
                - unix_timestamp(col("feedback_time"))
            ) / 60
        ).cast("int").alias("delay_minutes"),

        col("payload.confidence_before_score")
            .alias("confidence_before_score"),

        col("payload.perceived_understanding_before_score")
            .alias("perceived_understanding_before_score"),

        col("payload.expected_difficulty_score")
            .alias("expected_difficulty_score"),

        trim(col("payload.free_text_before"))
            .alias("free_text_before")
    )
)

pre_practice_feedback_silver_df.show(truncate=False)
pre_practice_feedback_silver_df.printSchema()

+------------+--------+-----------+------------+-------------------+-------------------+-------------+-----------------------+------------------------------------+-------------------------+------------------------------------------------------------------------------+
|feedback_id |user_id |session_id |practice_id |feedback_time      |ingestion_time     |delay_minutes|confidence_before_score|perceived_understanding_before_score|expected_difficulty_score|free_text_before                                                              |
+------------+--------+-----------+------------+-------------------+-------------------+-------------+-----------------------+------------------------------------+-------------------------+------------------------------------------------------------------------------+
|feedback_001|user_001|session_001|practice_001|2026-07-20 09:04:00|2026-07-20 09:04:03|0            |4                      |3                                   |8                        |I am

In [42]:
spark.table(
    "demo.silver.pre_practice_feedback"
).count()

0

In [43]:
pre_practice_feedback_silver_df.writeTo(
    "demo.silver.pre_practice_feedback"
).append()

In [44]:
spark.sql("""
SELECT COUNT(*) AS row_count
FROM demo.silver.pre_practice_feedback
""").show()

+---------+
|row_count|
+---------+
|        3|
+---------+



In [45]:
post_practice_feedback_silver_df = (
    learning_feedback_parsed_df
    .filter(col("feedback_stage") == "after_practice")
    .select(
        col("feedback_id"),
        col("user_id"),
        col("session_id"),
        col("practice_id"),
        col("feedback_time"),
        col("ingestion_time"),

        spark_round(
            (
                unix_timestamp(col("ingestion_time"))
                - unix_timestamp(col("feedback_time"))
            ) / 60
        ).cast("int").alias("delay_minutes"),

        col("payload.confidence_after_score")
            .alias("confidence_after_score"),

        col("payload.perceived_understanding_after_score")
            .alias("perceived_understanding_after_score"),

        col("payload.perceived_difficulty_score")
            .alias("perceived_difficulty_score"),

        col("payload.still_confused")
            .alias("still_confused"),

        trim(col("payload.free_text_after"))
            .alias("free_text_after")
    )
)

post_practice_feedback_silver_df.show(truncate=False)
post_practice_feedback_silver_df.printSchema()

+------------+--------+-----------+------------+-------------------+-------------------+-------------+----------------------+-----------------------------------+--------------------------+--------------+-----------------------------------------------------------------------------------+
|feedback_id |user_id |session_id |practice_id |feedback_time      |ingestion_time     |delay_minutes|confidence_after_score|perceived_understanding_after_score|perceived_difficulty_score|still_confused|free_text_after                                                                    |
+------------+--------+-----------+------------+-------------------+-------------------+-------------+----------------------+-----------------------------------+--------------------------+--------------+-----------------------------------------------------------------------------------+
|feedback_002|user_001|session_001|practice_001|2026-07-20 09:18:00|2026-07-20 09:22:00|4            |5                     |5          

In [46]:
spark.table(
    "demo.silver.post_practice_feedback"
).count()

0

In [47]:
post_practice_feedback_silver_df.writeTo(
    "demo.silver.post_practice_feedback"
).append()

In [48]:
spark.sql("""
SELECT COUNT(*) AS row_count
FROM demo.silver.post_practice_feedback
""").show()

+---------+
|row_count|
+---------+
|        3|
+---------+



In [49]:
learner_check_in_silver_df = (
    learning_feedback_parsed_df
    .filter(col("feedback_stage") == "general_check_in")
    .select(
        col("feedback_id"),
        col("user_id"),
        col("session_id"),
        col("feedback_time"),
        col("ingestion_time"),

        spark_round(
            (
                unix_timestamp(col("ingestion_time"))
                - unix_timestamp(col("feedback_time"))
            ) / 60
        ).cast("int").alias("delay_minutes"),

        col("payload.overall_confidence_score")
            .alias("overall_confidence_score"),

        col("payload.overall_motivation_score")
            .alias("overall_motivation_score"),

        col("payload.overall_stress_score")
            .alias("overall_stress_score"),

        trim(col("payload.free_text"))
            .alias("free_text")
    )
)

learner_check_in_silver_df.show(truncate=False)
learner_check_in_silver_df.printSchema()

+------------+--------+-----------+-------------------+-------------------+-------------+------------------------+------------------------+--------------------+------------------------------------------------------------------------------+
|feedback_id |user_id |session_id |feedback_time      |ingestion_time     |delay_minutes|overall_confidence_score|overall_motivation_score|overall_stress_score|free_text                                                                     |
+------------+--------+-----------+-------------------+-------------------+-------------+------------------------+------------------------+--------------------+------------------------------------------------------------------------------+
|feedback_007|user_001|session_001|2026-07-23 18:00:00|2026-07-23 18:04:00|4            |5                       |6                       |7                   |I feel more comfortable with recursion, but virtual memory is still difficult.|
|feedback_008|user_002|session_002|2026-

In [50]:
spark.table(
    "demo.silver.learner_check_in"
).count()

0

In [51]:
learner_check_in_silver_df.writeTo(
    "demo.silver.learner_check_in"
).append()

In [52]:
spark.sql("""
SELECT COUNT(*) AS row_count
FROM demo.silver.learner_check_in
""").show()

+---------+
|row_count|
+---------+
|        3|
+---------+



In [53]:
from pyspark.sql.functions import explode

learner_check_in_topics_silver_df = (
    learning_feedback_parsed_df
    .filter(col("feedback_stage") == "general_check_in")
    .withColumn(
        "topic_feedback",
        explode(col("payload.topics_feedback"))
    )
    .select(
        col("feedback_id"),
        col("user_id"),
        col("session_id"),
        col("topic_feedback.topic_id").alias("topic_id"),
        col("feedback_time"),
        col("topic_feedback.perceived_understanding_score")
            .alias("perceived_understanding_score"),
        col("topic_feedback.confidence_score")
            .alias("topic_confidence_score"),
        col("topic_feedback.still_confused")
            .alias("still_confused")
    )
)

learner_check_in_topics_silver_df.show(truncate=False)
learner_check_in_topics_silver_df.printSchema()

+------------+--------+-----------+--------------------+-------------------+-----------------------------+----------------------+--------------+
|feedback_id |user_id |session_id |topic_id            |feedback_time      |perceived_understanding_score|topic_confidence_score|still_confused|
+------------+--------+-----------+--------------------+-------------------+-----------------------------+----------------------+--------------+
|feedback_007|user_001|session_001|topic_virtual_memory|2026-07-23 18:00:00|4                            |4                     |true          |
|feedback_007|user_001|session_001|topic_recursion     |2026-07-23 18:00:00|7                            |7                     |false         |
|feedback_008|user_002|session_002|topic_recursion     |2026-07-24 17:30:00|5                            |5                     |true          |
|feedback_009|user_003|session_003|topic_memory_layout |2026-07-25 16:00:00|9                            |9                     |f

In [54]:
spark.table(
    "demo.silver.learner_check_in"
).count()

3

In [55]:
learner_check_in_silver_df.writeTo(
    "demo.silver.learner_check_in"
).append()

In [56]:
spark.table(
    "demo.silver.learner_check_in_topics"
).count()

0

In [57]:
learner_check_in_topics_silver_df.writeTo(
    "demo.silver.learner_check_in_topics"
).append()

In [58]:
spark.sql("""
SELECT COUNT(*) AS row_count
FROM demo.silver.learner_check_in
""").show()

spark.sql("""
SELECT COUNT(*) AS row_count
FROM demo.silver.learner_check_in_topics
""").show()

+---------+
|row_count|
+---------+
|        6|
+---------+

+---------+
|row_count|
+---------+
|        5|
+---------+



In [59]:
spark.sql("""
DELETE FROM demo.silver.learner_check_in
""")

DataFrame[]

In [60]:
spark.table(
    "demo.silver.learner_check_in"
).count()

0

In [61]:
learner_check_in_silver_df.writeTo(
    "demo.silver.learner_check_in"
).append()

In [62]:
spark.sql("""
SELECT COUNT(*) AS row_count
FROM demo.silver.learner_check_in
""").show()

spark.sql("""
SELECT COUNT(*) AS row_count
FROM demo.silver.learner_check_in_topics
""").show()

+---------+
|row_count|
+---------+
|        3|
+---------+

+---------+
|row_count|
+---------+
|        5|
+---------+



In [63]:
from pyspark.sql.functions import (
    col,
    from_json,
    explode,
    concat_ws,
    sha2,
    trim,
    lower,
    lit
)

ai_interactions_df = (
    learning_events_bronze_df
    .filter(col("event_type") == "ai_learning_interaction")
    .withColumn(
        "parsed_payload",
        from_json(
            col("raw_payload"),
            learning_event_payload_schema
        )
    )
    .withColumn(
        "dynamic_concept_name",
        explode(col("parsed_payload.detected_concepts"))
    )
)

In [64]:
ai_extracted_insights_silver_df = (
    ai_interactions_df
    .withColumn(
        "insight_id",
        sha2(
            concat_ws(
                "||",
                col("event_id"),
                col("dynamic_concept_name")
            ),
            256
        )
    )
    .select(
        col("insight_id"),
        col("event_id"),
        col("user_id"),
        col("session_id"),
        col("ingestion_time").alias("extracted_at"),
        trim(col("dynamic_concept_name"))
            .alias("dynamic_concept_name"),

        lit(1.0).cast("float")
            .alias("extraction_confidence"),

        trim(col("parsed_payload.processing_model"))
            .alias("processing_model"),

        lit("pending")
            .alias("validation_status"),

        col("raw_payload")
            .alias("ai_attributes")
    )
)

ai_extracted_insights_silver_df.show(truncate=False)
ai_extracted_insights_silver_df.printSchema()

+----------------------------------------------------------------+--------+--------+-----------+-------------------+--------------------+---------------------+------------------------+-----------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|insight_id          

In [65]:
ai_extracted_insights_silver_df.count()

9

In [66]:
spark.table(
    "demo.silver.ai_extracted_insights"
).count()

0

In [67]:
ai_extracted_insights_silver_df.writeTo(
    "demo.silver.ai_extracted_insights"
).append()

In [68]:
spark.sql("""
SELECT COUNT(*) AS row_count
FROM demo.silver.ai_extracted_insights
""").show()

+---------+
|row_count|
+---------+
|        9|
+---------+



In [69]:
spark.sql("""
SELECT
    insight_id,
    event_id,
    user_id,
    dynamic_concept_name,
    extraction_confidence,
    processing_model,
    validation_status
FROM demo.silver.ai_extracted_insights
ORDER BY event_id, dynamic_concept_name
""").show(truncate=False)

+----------------------------------------------------------------+--------+--------+--------------------+---------------------+------------------------+-----------------+
|insight_id                                                      |event_id|user_id |dynamic_concept_name|extraction_confidence|processing_model        |validation_status|
+----------------------------------------------------------------+--------+--------+--------------------+---------------------+------------------------+-----------------+
|a812e708876c32b4bb8d415842bf2de7a9114b7b88efe690f0c36a66875f65df|evt_0001|user_001|Operating Systems   |1.0                  |synthetic_ai_analysis_v1|pending          |
|a6cb2c3a4bd19f4a389472160bd13969533dbe49e042f7855ab5ac61c66bfab6|evt_0001|user_001|Page Fault          |1.0                  |synthetic_ai_analysis_v1|pending          |
|66317e7e2b57674ab50b4418ee576ccc39e490ca3f3f5009cd3ad2c3d474a3fe|evt_0001|user_001|Virtual Memory      |1.0                  |synthetic_ai_analy